In [6]:
import lightgbm as lgb
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

In [7]:
# 1. データの読み込みとカテゴリ型変換
train_df = pd.read_csv("../../data/raw/train.csv")

drop_cols = ["RowNumber", "CustomerId", "Surname", "id"]
existing_drop_cols = [col for col in drop_cols if col in train_df.columns]
train_df = train_df.drop(columns=existing_drop_cols)

for col in train_df.columns:
    if not pd.api.types.is_numeric_dtype(train_df[col]):
        train_df[col] = train_df[col].astype("category")

In [8]:
# 2. 特徴量と正解に分ける
X = train_df.drop(columns=["Exited"])
y = train_df["Exited"]

In [9]:
# 3. 学習データと検証データに分割
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
import os
import joblib  # 👈 モデル保存のために使用
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

# 必要モデルのインポート
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

# --- 前段階：カテゴリ変数の特定とRandom Forest用の準備 ---
cat_cols = [col for col in X.columns if X[col].dtype.name == "category"]

X_rf = X.copy()
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
if cat_cols:
    X_rf[cat_cols] = oe.fit_transform(X_rf[cat_cols].astype(str))

# --- データの分割 ---
X_train_g, X_val_g, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_r, X_val_r, _, _ = train_test_split(
    X_rf, y, test_size=0.2, random_state=42, stratify=y
)


# --- 4つのモデルの作成と学習 ---
print("1/4: LightGBM を学習中...")
model_lgb = lgb.LGBMClassifier(random_state=42, verbose=-1)
model_lgb.fit(X_train_g, y_train)
pred_lgb = model_lgb.predict_proba(X_val_g)[:, 1]

print("2/4: XGBoost を学習中...")
model_xgb = xgb.XGBClassifier(random_state=42, enable_categorical=True, eval_metric='logloss')
model_xgb.fit(X_train_g, y_train)
pred_xgb = model_xgb.predict_proba(X_val_g)[:, 1]

print("3/4: CatBoost を学習中...")
X_train_c = X_train_g.copy()
X_val_c = X_val_g.copy()
for col in cat_cols:
    X_train_c[col] = X_train_c[col].astype(str)
    X_val_c[col] = X_val_c[col].astype(str)

model_cat = CatBoostClassifier(random_state=42, verbose=0, cat_features=cat_cols)
model_cat.fit(X_train_c, y_train)
pred_cat = model_cat.predict_proba(X_val_c)[:, 1]

print("4/4: Random Forest を学習中...")
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train_r, y_train)
pred_rf = model_rf.predict_proba(X_val_r)[:, 1]

print("-" * 40)
print("すべてのモデルの学習が完了しました！")
print("-" * 40)


# --- 🎉 各モデルの保存処理 (追加部分) ---
# 保存先ディレクトリの作成（存在しない場合は自動作成されます）
save_dir = "../../model/"
os.makedirs(save_dir, exist_ok=True)

# プレフィックスの定義
prefix = "05251045_"

print("モデルを保存しています...")
joblib.dump(model_lgb, f"{save_dir}{prefix}lightgbm.pkl")
joblib.dump(model_xgb, f"{save_dir}{prefix}xgboost.pkl")
joblib.dump(model_cat, f"{save_dir}{prefix}catboost.pkl")
joblib.dump(model_rf, f"{save_dir}{prefix}randomforest.pkl")
print("👉 すべてのモデルを ../../model/ に保存しました！")
print("-" * 40)


# --- 各モデルのROC AUCを表示 ---
auc_lgb = roc_auc_score(y_val, pred_lgb)
auc_xgb = roc_auc_score(y_val, pred_xgb)
auc_cat = roc_auc_score(y_val, pred_cat)
auc_rf = roc_auc_score(y_val, pred_rf)

print("【検証データに対する ROC AUC スコア】")
print(f"LightGBM     : {auc_lgb:.5f}")
print(f"XGBoost      : {auc_xgb:.5f}")
print(f"CatBoost     : {auc_cat:.5f}")
print(f"RandomForest : {auc_rf:.5f}")
print("-" * 40)


# --- 可視化・分析用のデータフレーム（df_errors）を作成 ---
df_errors = X_val_g.copy()
df_errors["Actual"] = y_val.values

df_errors["Pred_LightGBM"] = pred_lgb
df_errors["Pred_XGBoost"] = pred_xgb
df_errors["Pred_CatBoost"] = pred_cat
df_errors["Pred_RandomForest"] = pred_rf

threshold = 0.5
df_errors["Miss_LightGBM"] = ((df_errors["Pred_LightGBM"] >= threshold).astype(int) != df_errors["Actual"]).astype(int)
df_errors["Miss_XGBoost"] = ((df_errors["Pred_XGBoost"] >= threshold).astype(int) != df_errors["Actual"]).astype(int)
df_errors["Miss_CatBoost"] = ((df_errors["Pred_CatBoost"] >= threshold).astype(int) != df_errors["Actual"]).astype(int)
df_errors["Miss_RandomForest"] = ((df_errors["Pred_RandomForest"] >= threshold).astype(int) != df_errors["Actual"]).astype(int)

df_errors["Total_Miss_Count"] = df_errors[["Miss_LightGBM", "Miss_XGBoost", "Miss_CatBoost", "Miss_RandomForest"]].sum(axis=1)

# CSVファイルとして保存
df_errors.to_csv("../../results/val_predictions_comparison.csv", index=False)
print("詳細データを '../../results/val_predictions_comparison.csv' に保存しました。")

1/4: LightGBM を学習中...
2/4: XGBoost を学習中...
3/4: CatBoost を学習中...
4/4: Random Forest を学習中...
----------------------------------------
すべてのモデルの学習が完了しました！
----------------------------------------
モデルを保存しています...
👉 すべてのモデルを ../../model/ に保存しました！
----------------------------------------
【検証データに対する ROC AUC スコア】
LightGBM     : 0.93089
XGBoost      : 0.92483
CatBoost     : 0.93417
RandomForest : 0.92522
----------------------------------------
詳細データを 'val_predictions_comparison.csv' に保存しました。
